Реализация hnm для локального запуска без необходимости обучать модель

In [25]:
import os
import csv
import torch
import logging
from tqdm import tqdm

import hydra
from hydra import initialize, compose
from hydra.utils import instantiate
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

import rootutils


In [26]:
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

In [27]:
GlobalHydra.instance().clear()

rootutils.setup_root(
    search_from=".",
    indicator=".project-root",
    pythonpath=True,
)


WindowsPath('D:/ocr-project/modeling_recognizer')

In [33]:
with initialize(version_base="1.3", config_path="../configs"):
    cfg = compose(
        config_name="train.yaml",
        overrides=[
            "train=false",
            "test=false",
            "experiment=baseline_parseq",
        ],
    )

print(OmegaConf.to_yaml(cfg))


task_name: train
experiment_name: baseline_parseq
tags:
- parseq
- baseline
- ocr_dataset_v3
train: false
test: false
ckpt_path: null
seed: 12345
data:
  _target_: src.data.ocr_datamodule.OCRDataModule
  data_path: ${paths.data_dir}
  dataset: ${paths.data_dir}/ocr_dataset_v3_31_len.parquet
  vocab: 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyzЁІАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёіҒғҚқҢңҮүҰұҺһӘәӨө0123456789!"#$%&)*+,-./\\:;<=>?@()‚„•°[]^_`{|}~«»“”’—–₸₽№£ '
  timesteps: 32
  batch_size: 128
  num_workers: 4
  train_frac: 0.85
  pin_memory: true
  persistent_workers: true
  input_shape:
  - 3
  - 32
  - 128
  augs: hard
model:
  _target_: src.models.parseq.PARSeqModel
  vocab: ${data.vocab}
  input_shape: ${data.input_shape}
  pretrained: true
  weights_path: ..\weights\parseq_baseline__2026_02_09.pth
  compile: false
  optimizer:
    _target_: torch.optim.AdamW
    _partial_: true
    lr: 0.001
    weight_decay: 0.0005
    betas:
    - 0.9
    - 0.9

In [34]:
log.info(f"Instantiating datamodule <{cfg.data._target_}>")

datamodule = instantiate(cfg.data)

datamodule.prepare_data()
datamodule.setup("fit")

eval_loader = datamodule.val_dataloader()
train_loader = datamodule.train_dataloader()

if eval_loader is None and train_loader is None:
    raise RuntimeError("No train_loader and no eval_loader provided")

[RandomApply(transform=ColorInversion(min_val=0.5), p=0.2), RandomGrayscale(p=0.25), RandomPhotometricDistort(p=0.2), RandomApply(transform=RandomShadow(opacity_range=(0.2, 0.8)), p=0.2), RandomApply(transform=GaussianNoise(mean=0.0, std=0.1), p=0.3), RandomApply(transform=RandomApplyJpeg(), p=0.4), RandomApply(transform=RandomRotation(degrees=[-3.0, 3.0], interpolation=InterpolationMode.NEAREST, expand=False, fill=0), p=0.4), RandomPerspective(p=0.6, distortion_scale=0.1, interpolation=InterpolationMode.BILINEAR, fill=0), RandomErasing(p=0.25, value=[0.0], inplace=False)]
Compose(
      ToImage()
      Resize(output_size=(32, 128), interpolation='bilinear', preserve_aspect_ratio=True, symmetric_pad=False)
      RandomApply(transform=ColorInversion(min_val=0.5), p=0.2)
      RandomGrayscale(p=0.25)
      RandomPhotometricDistort(p=0.2)
      RandomApply(transform=RandomShadow(opacity_range=(0.2, 0.8)), p=0.2)
      RandomApply(transform=GaussianNoise(mean=0.0, std=0.1), p=0.3)
      Ra

In [37]:
log.info(f"Instantiating model <{cfg.model._target_}>")

model = instantiate(cfg.model)

ckpt_path = None #cfg.model.get("weights_path") or cfg.get("ckpt_path")
#if ckpt_path is None:
    #raise RuntimeError("No checkpoint provided (load_weights / ckpt_path)")

# state = torch.load(ckpt_path, map_location="cpu")

# if "state_dict" in state:
#     model.load_state_dict(state["state_dict"])
# else:
#     model.load_state_dict(state)

model.eval()
model.cuda()

device = next(model.parameters()).device


In [38]:
datamodule.prepare_data()
datamodule.setup("fit")

eval_loader = datamodule.val_dataloader()
train_loader = datamodule.train_dataloader()

if eval_loader is None and train_loader is None:
    log.info("⚠️ HNM: no train_loader and no eval_loader provided")


In [40]:
output_dir = 'hnms'
os.makedirs(output_dir, exist_ok=True)

csv_path_val = os.path.join(output_dir, "val_hnm.csv")
results = []

log.info("🔍 Collecting hard negatives (val)...")

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(eval_loader)):
        images, labels, paths = batch

        for sample_idx, (img, gt, path) in enumerate(zip(images, labels, paths)):
            img = img.unsqueeze(0).to(device)

            output = model((img, [gt], path))

            preds = output.get("preds", [()])
            loss_val = output.get("loss", None)

            for pred_text, conf in preds:
                entry = {
                    "batch_idx": batch_idx,
                    "sample_idx": sample_idx,
                    "path": path,
                    "pred_text": pred_text,
                    "confidence": float(conf),
                    "loss": loss_val,
                    "gt": gt,
                }
                results.append(entry)


  4%|▍         | 43/1123 [03:50<1:36:24,  5.36s/it]


KeyboardInterrupt: 

In [ ]:
if len(results) == 0:
    log.warning("No validation HNM results collected")
else:
    with open(csv_path_val, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    log.info(f"✅ Hard negatives saved to: {csv_path_val}")


In [ ]:
csv_path_train = os.path.join(output_dir, "train_hnm.csv")
results = []

log.info("🔍 Collecting hard negatives (train)...")

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(train_loader)):
        images, labels, paths = batch

        for sample_idx, (img, gt, path) in enumerate(zip(images, labels, paths)):
            img = img.unsqueeze(0).to(device)

            output = model((img, [gt], path))

            preds = output.get("preds", [()])
            loss_val = output.get("loss", None)

            for pred_text, conf in preds:
                entry = {
                    "batch_idx": batch_idx,
                    "sample_idx": sample_idx,
                    "path": path,
                    "pred_text": pred_text,
                    "confidence": float(conf),
                    "loss": loss_val,
                    "gt": gt,
                }
                results.append(entry)

In [ ]:
if len(results) == 0:
    log.warning("No train HNM results collected")
else:
    with open(csv_path_train, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

    log.info(f"✅ Hard negatives saved to: {csv_path_train}")